# IEX Data Cleaner

Notebook này sẽ xử lý file `iex-data.xlsx` để đưa về dạng chuẩn: Agent | Activity | Start | End.
- Xử lý phần header dư thừa
- Chuẩn hóa tên cột
- Chuyển Start/End về datetime
- Xuất ra file sạch `iex_clean.csv`


In [62]:
import pandas as pd

# Hiển thị đủ số dòng và số cột mong muốn
pd.set_option("display.max_rows", 100)   # tối đa số dòng hiển thị
pd.set_option("display.max_columns", None)  # hiện tất cả các cột
pd.set_option("display.width", None)   # không giới hạn độ rộng

# Đọc file, bỏ 5 dòng đầu tiên
iex_df = pd.read_excel("iex-data.xlsx", skiprows=5)

# Bỏ những dòng toàn NaN
iex_df = iex_df.dropna(how="all")

# Xóa các cột toàn NaN
iex_df = iex_df.dropna(axis=1, how="all")

# Reset lại index
iex_df = iex_df.reset_index(drop=True)

# Xem thử 50 dòng đầu tiên
iex_df.head(50)

c:\Users\huuchinh.nguyen\AppData\Local\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,MU: 1334 Expedia ENG Ho Chi Minh VNM,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 10
0,"Agent: 3052306 BUI, BADUONG",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Date,Start,End,Scheduled Activity,NaN,Start,End
2,NaN,8/26/25,1:00 PM,10:00 PM,NaN,Open Time,1:00 PM,1:45 PM
3,NaN,NaN,NaN,NaN,NaN,Break,1:45 PM,2:00 PM
4,NaN,NaN,NaN,NaN,NaN,Open Time,2:00 PM,5:45 PM
5,NaN,NaN,NaN,NaN,NaN,Break,5:45 PM,6:00 PM
6,NaN,NaN,NaN,NaN,NaN,Open Time,6:00 PM,8:00 PM
7,NaN,NaN,NaN,NaN,NaN,Lunch,8:00 PM,9:00 PM
8,NaN,NaN,NaN,NaN,NaN,Open Time,9:00 PM,10:00 PM
9,NaN,8/27/25,1:00 PM,10:00 PM,NaN,Open Time,1:00 PM,2:10 PM


In [63]:
# Gán lại tên cột thành số (0, 1, 2, ...)
iex_df.columns = range(iex_df.shape[1])

# Xem thử 20 dòng đầu
iex_df.head(20)

,0,1,2,3,4,5,6,7
0,"Agent: 3052306 BUI, BADUONG",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Date,Start,End,Scheduled Activity,NaN,Start,End
2,NaN,8/26/25,1:00 PM,10:00 PM,NaN,Open Time,1:00 PM,1:45 PM
3,NaN,NaN,NaN,NaN,NaN,Break,1:45 PM,2:00 PM
4,NaN,NaN,NaN,NaN,NaN,Open Time,2:00 PM,5:45 PM
5,NaN,NaN,NaN,NaN,NaN,Break,5:45 PM,6:00 PM
6,NaN,NaN,NaN,NaN,NaN,Open Time,6:00 PM,8:00 PM
7,NaN,NaN,NaN,NaN,NaN,Lunch,8:00 PM,9:00 PM
8,NaN,NaN,NaN,NaN,NaN,Open Time,9:00 PM,10:00 PM
9,NaN,8/27/25,1:00 PM,10:00 PM,NaN,Open Time,1:00 PM,2:10 PM


In [64]:
import re

# Hàm tách IEX Id và Name từ chuỗi
def split_agent_info(value):
    if isinstance(value, str) and value.startswith("Agent:"):
        # Loại bỏ "Agent:" rồi strip
        cleaned = value.replace("Agent:", "").strip()
        # Dùng regex tách id (số đầu tiên) và phần còn lại (tên)
        match = re.match(r"(\d+)\s+(.+)", cleaned)
        if match:
            return match.group(1), match.group(2)
    return None, None

# Tách thông tin Agent
iex_df["IEX Id"], iex_df["Name"] = zip(*iex_df[0].map(split_agent_info))

# Kiểm tra kết quả
iex_df.head(20)

,0,1,2,3,4,5,6,7,IEX Id,Name
0,"Agent: 3052306 BUI, BADUONG",NaN,NaN,NaN,NaN,NaN,NaN,NaN,3052306,"BUI, BADUONG"
1,NaN,Date,Start,End,Scheduled Activity,NaN,Start,End,None,None
2,NaN,8/26/25,1:00 PM,10:00 PM,NaN,Open Time,1:00 PM,1:45 PM,None,None
3,NaN,NaN,NaN,NaN,NaN,Break,1:45 PM,2:00 PM,None,None
4,NaN,NaN,NaN,NaN,NaN,Open Time,2:00 PM,5:45 PM,None,None
5,NaN,NaN,NaN,NaN,NaN,Break,5:45 PM,6:00 PM,None,None
6,NaN,NaN,NaN,NaN,NaN,Open Time,6:00 PM,8:00 PM,None,None
7,NaN,NaN,NaN,NaN,NaN,Lunch,8:00 PM,9:00 PM,None,None
8,NaN,NaN,NaN,NaN,NaN,Open Time,9:00 PM,10:00 PM,None,None
9,NaN,8/27/25,1:00 PM,10:00 PM,NaN,Open Time,1:00 PM,2:10 PM,None,None


In [65]:
# 1. Xóa cột 0
iex_df = iex_df.drop(columns=[0])

# 2. Đưa IEX Id và Name ra ngoài cùng bên trái
cols = ["IEX Id", "Name"] + [col for col in iex_df.columns if col not in ["IEX Id", "Name"]]
iex_df = iex_df[cols]

# 3. Nếu cột 2 = "Off" thì cột 3 = "Off"
iex_df.loc[iex_df[2] == "Off", 3] = "Off"

# Xem kết quả
iex_df.head(20)

,IEX Id,Name,1,2,3,4,5,6,7
0,3052306,"BUI, BADUONG",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,None,None,Date,Start,End,Scheduled Activity,NaN,Start,End
2,None,None,8/26/25,1:00 PM,10:00 PM,NaN,Open Time,1:00 PM,1:45 PM
3,None,None,NaN,NaN,NaN,NaN,Break,1:45 PM,2:00 PM
4,None,None,NaN,NaN,NaN,NaN,Open Time,2:00 PM,5:45 PM
5,None,None,NaN,NaN,NaN,NaN,Break,5:45 PM,6:00 PM
6,None,None,NaN,NaN,NaN,NaN,Open Time,6:00 PM,8:00 PM
7,None,None,NaN,NaN,NaN,NaN,Lunch,8:00 PM,9:00 PM
8,None,None,NaN,NaN,NaN,NaN,Open Time,9:00 PM,10:00 PM
9,None,None,8/27/25,1:00 PM,10:00 PM,NaN,Open Time,1:00 PM,2:10 PM


In [66]:
# Tạo cột Shift
def get_shift(row):
    # Nếu cột 1 (Name) bị NaN hoặc bằng "Date" thì bỏ qua
    if pd.isna(row[1]) or str(row[1]).strip().lower() == "date":
        return None
    # Nếu cả Start và End đều là Off
    elif str(row[2]).strip().lower() == "off" and str(row[3]).strip().lower() == "off":
        return "Off"
    # Nếu có giờ Start và End
    elif pd.notna(row[2]) and pd.notna(row[3]):
        return f"{row[2]} - {row[3]}"
    else:
        return None

iex_df["Shift"] = iex_df.apply(get_shift, axis=1)

# Đưa cột Shift ngay sau cột Name
cols = list(iex_df.columns)
name_index = cols.index("Name")
cols.insert(name_index + 1, cols.pop(cols.index("Shift")))
iex_df = iex_df[cols]

iex_df.head(50)


,IEX Id,Name,Shift,1,2,3,4,5,6,7
0,3052306,"BUI, BADUONG",None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,None,None,None,Date,Start,End,Scheduled Activity,NaN,Start,End
2,None,None,1:00 PM - 10:00 PM,8/26/25,1:00 PM,10:00 PM,NaN,Open Time,1:00 PM,1:45 PM
3,None,None,None,NaN,NaN,NaN,NaN,Break,1:45 PM,2:00 PM
4,None,None,None,NaN,NaN,NaN,NaN,Open Time,2:00 PM,5:45 PM
5,None,None,None,NaN,NaN,NaN,NaN,Break,5:45 PM,6:00 PM
6,None,None,None,NaN,NaN,NaN,NaN,Open Time,6:00 PM,8:00 PM
7,None,None,None,NaN,NaN,NaN,NaN,Lunch,8:00 PM,9:00 PM
8,None,None,None,NaN,NaN,NaN,NaN,Open Time,9:00 PM,10:00 PM
9,None,None,1:00 PM - 10:00 PM,8/27/25,1:00 PM,10:00 PM,NaN,Open Time,1:00 PM,2:10 PM


In [67]:
# Downfill IEX Id và Name trước
iex_df["IEX Id"] = iex_df["IEX Id"].ffill()
iex_df["Name"] = iex_df["Name"].ffill()

# Shift: vừa upfill vừa downfill trong phạm vi từng IEX Id
iex_df["Shift"] = (
    iex_df.groupby("IEX Id")["Shift"]
    .transform(lambda x: x.ffill().bfill())
)


# Xử lý cột 1 (ngày): thay "Date" bằng NaN, rồi upfill + downfill trong phạm vi IEX Id
iex_df[1] = iex_df[1].replace("Date", pd.NA)
iex_df[1] = (
    iex_df.groupby("IEX Id")[1]
    .transform(lambda x: x.ffill().bfill())
)

# Đổi tên cột 1 thành "Date"
iex_df = iex_df.rename(columns={1: "Date"})

# Chuẩn hoá format ngày (YYYY-MM-DD)
iex_df["Date"] = pd.to_datetime(iex_df["Date"], errors="coerce").dt.strftime("%Y-%m-%d")

# Remove cột 2,3 (giữ lại cột "Date")
iex_df = iex_df.drop(columns=[2, 3])

# Remove dòng Scheduled Activity
iex_df = iex_df[iex_df[4] != "Scheduled Activity"]

# Remove dòng mà 5,6,7 đều NaN
iex_df = iex_df.dropna(subset=[5, 6, 7], how="all")

iex_df


C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_20684\3251114315.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  iex_df["Date"] = pd.to_datetime(iex_df["Date"], errors="coerce").dt.strftime("%Y-%m-%d")


,IEX Id,Name,Shift,Date,4,5,6,7
2,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,NaN,Open Time,1:00 PM,1:45 PM
3,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,NaN,Break,1:45 PM,2:00 PM
4,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,NaN,Open Time,2:00 PM,5:45 PM
5,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,NaN,Break,5:45 PM,6:00 PM
6,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,NaN,Open Time,6:00 PM,8:00 PM
...,...,...,...,...,...,...,...,...
2235,3089155,"Vuu, TienNhan",5:00 AM - 2:00 PM,2025-08-27,NaN,Open Time,6:15 AM,9:10 AM
2236,3089155,"Vuu, TienNhan",5:00 AM - 2:00 PM,2025-08-27,NaN,Lunch,9:10 AM,10:10 AM
2237,3089155,"Vuu, TienNhan",5:00 AM - 2:00 PM,2025-08-27,NaN,Open Time,10:10 AM,12:30 PM
2238,3089155,"Vuu, TienNhan",5:00 AM - 2:00 PM,2025-08-27,NaN,Break,12:30 PM,12:45 PM


In [68]:
# Xóa cột 4
iex_df = iex_df.drop(columns=[4])

# Đổi tên cột 5,6,7
iex_df = iex_df.rename(columns={
    5: "Activity",
    6: "Start time",
    7: "End time"
})

iex_df.head(50)

,IEX Id,Name,Shift,Date,Activity,Start time,End time
2,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Open Time,1:00 PM,1:45 PM
3,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Break,1:45 PM,2:00 PM
4,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Open Time,2:00 PM,5:45 PM
5,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Break,5:45 PM,6:00 PM
6,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Open Time,6:00 PM,8:00 PM
7,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Lunch,8:00 PM,9:00 PM
8,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Open Time,9:00 PM,10:00 PM
9,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-27,Open Time,1:00 PM,2:10 PM
10,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-27,Break,2:10 PM,2:25 PM
11,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-27,Open Time,2:25 PM,4:00 PM


In [69]:
# -----------------------
# Fix rollover / cross-midnight robust
# -----------------------
from datetime import datetime, timedelta
import pandas as pd

# 1. Chuẩn hoá cột Date / Start time / End time
iex_df['Date'] = pd.to_datetime(iex_df['Date'], errors='coerce').dt.normalize()  # normalize -> midnight datetime
# NOTE: we keep Start/End as-is to extract time part (they may be strings or datetimes)
iex_df['Start time'] = pd.to_datetime(iex_df['Start time'], errors='coerce')
iex_df['End time']   = pd.to_datetime(iex_df['End time'], errors='coerce')

# helper: lấy time-of-day (datetime.time) từ giá trị có thể là datetime hoặc string
def _get_time_of_day(val):
    if pd.isna(val):
        return None
    if isinstance(val, (pd.Timestamp, datetime)):
        return val.time()
    try:
        # cố parse chuỗi thành datetime rồi lấy .time()
        return pd.to_datetime(str(val), errors='coerce').time()
    except Exception:
        return None

# Chọn groupby keys, nếu Shift không tồn tại thì dùng IEX Id + Name
group_keys = ['IEX Id', 'Name', 'Shift'] if 'Shift' in iex_df.columns else ['IEX Id', 'Name']

new_rows = []

# duyệt theo nhóm, giữ thứ tự index gốc (không sort theo time-of-day)
for _, group in iex_df.groupby(group_keys, sort=False):
    # giữ lại thứ tự gốc của group
    group = group.sort_index()
    # base_date: lấy từ cột Date của dòng đầu group, nếu null thì fallback lấy ngày từ Start time hoặc hôm nay
    first_date = group['Date'].iloc[0]
    if pd.isna(first_date):
        # thử lấy ngày từ first Start time
        first_start = group['Start time'].dropna()
        if not first_start.empty:
            base_date = first_start.iloc[0].date()
        else:
            base_date = pd.Timestamp.now().date()
    else:
        base_date = first_date.date() if isinstance(first_date, (pd.Timestamp, datetime)) else pd.to_datetime(first_date).date()
    
    day_offset = 0
    prev_time = None
    
    for idx, row in group.iterrows():
        st_t = _get_time_of_day(row['Start time'])
        en_t = _get_time_of_day(row['End time'])
        
        # Nếu không có start time, giữ nguyên (k vẫn append với NaT)
        if st_t is None:
            new_start = pd.NaT
        else:
            # nếu prev_time có giá trị và time-of-day hiện tại < prev_time => đã qua nửa đêm => tăng offset
            if prev_time is not None and st_t < prev_time:
                day_offset += 1
            new_start = datetime.combine(base_date + timedelta(days=day_offset), st_t)
        
        # Tính end: default end_offset = day_offset, nhưng nếu en_t < st_t (time-only) => end sang ngày sau
        if en_t is None:
            new_end = pd.NaT
        else:
            end_offset = day_offset
            if st_t is not None and en_t < st_t:
                end_offset = day_offset + 1
            new_end = datetime.combine(base_date + timedelta(days=end_offset), en_t)
        
        # cập nhật prev_time (dùng st_t nếu có, còn không giữ prev_time)
        if st_t is not None:
            prev_time = st_t
        
        new_row = row.copy()
        new_row['Start time'] = new_start
        new_row['End time']   = new_end
        # cập nhật Date theo Start time nếu Start time tồn tại
        if pd.notna(new_start):
            new_row['Date'] = pd.to_datetime(new_start).normalize()
        else:
            # giữ nguyên Date nếu không có Start
            new_row['Date'] = row['Date']
        
        new_rows.append(new_row)

# tạo DataFrame mới và reset index
iex_df = pd.DataFrame(new_rows).reset_index(drop=True)

# (tuỳ chọn) convert Date hiển thị thành string YYYY-MM-DD
# iex_df['Date'] = iex_df['Date'].dt.strftime('%Y-%m-%d')

# Kiểm tra nhanh: đảm bảo Start time date == Date
mismatch = iex_df.loc[iex_df['Start time'].notna() & (iex_df['Start time'].dt.normalize() != iex_df['Date'])]
print("Số dòng start-date != Date sau xử lý:", len(mismatch))

# hiển thị vài dòng quanh vị trí problem (nếu có)
if len(mismatch) > 0:
    display(mismatch.head(10))

# show sample
iex_df.head(60)

#iex_df[iex_df["Name"]=="Dang, PhuongTien"]

C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_20684\1771324730.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  iex_df['Start time'] = pd.to_datetime(iex_df['Start time'], errors='coerce')
C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_20684\1771324730.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  iex_df['End time']   = pd.to_datetime(iex_df['End time'], errors='coerce')


Số dòng start-date != Date sau xử lý: 0


,IEX Id,Name,Shift,Date,Activity,Start time,End time
0,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Open Time,2025-08-26 13:00:00,2025-08-26 13:45:00
1,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Break,2025-08-26 13:45:00,2025-08-26 14:00:00
2,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Open Time,2025-08-26 14:00:00,2025-08-26 17:45:00
3,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Break,2025-08-26 17:45:00,2025-08-26 18:00:00
4,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Open Time,2025-08-26 18:00:00,2025-08-26 20:00:00
5,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Lunch,2025-08-26 20:00:00,2025-08-26 21:00:00
6,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Open Time,2025-08-26 21:00:00,2025-08-26 22:00:00
7,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-27,Open Time,2025-08-27 13:00:00,2025-08-27 14:10:00
8,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-27,Break,2025-08-27 14:10:00,2025-08-27 14:25:00
9,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-27,Open Time,2025-08-27 14:25:00,2025-08-27 16:00:00


In [70]:
# --- Bổ sung thông tin từ Headcount master ---
# Đọc file dimension-headcount.xlsx
hc_df = pd.read_excel(
    r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Vu\unscheduled-agent\dimension-headcount.xlsx"
)

# Chuẩn hóa IEX Id trong cả 2 df
iex_df["IEX Id"] = iex_df["IEX Id"].astype(str).str.strip().str.split(".").str[0]
hc_df["IEX ID"] = hc_df["IEX ID"].astype(str).str.strip().str.split(".").str[0]

# Merge
iex_df = iex_df.merge(
    hc_df[["IEX ID", "Site", "Supervisor Name", "LOB", "Email Id"]],
    left_on="IEX Id",
    right_on="IEX ID",
    how="left"
).drop(columns=["IEX ID"])

iex_df.head(50)

,IEX Id,Name,Shift,Date,Activity,Start time,End time,Site,Supervisor Name,LOB,Email Id
0,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Open Time,2025-08-26 13:00:00,2025-08-26 13:45:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
1,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Break,2025-08-26 13:45:00,2025-08-26 14:00:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
2,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Open Time,2025-08-26 14:00:00,2025-08-26 17:45:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
3,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Break,2025-08-26 17:45:00,2025-08-26 18:00:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
4,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Open Time,2025-08-26 18:00:00,2025-08-26 20:00:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
5,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Lunch,2025-08-26 20:00:00,2025-08-26 21:00:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
6,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Open Time,2025-08-26 21:00:00,2025-08-26 22:00:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
7,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-27,Open Time,2025-08-27 13:00:00,2025-08-27 14:10:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
8,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-27,Break,2025-08-27 14:10:00,2025-08-27 14:25:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
9,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-27,Open Time,2025-08-27 14:25:00,2025-08-27 16:00:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com


In [71]:
# Thống kê số lượng agent không tìm thấy trong Headcount master
missing_ids = iex_df[iex_df["Site"].isna()]["IEX Id"].unique()

print(f"Số lượng IEX Id trong iex_df nhưng không tìm thấy trong headcount master: {len(missing_ids)}")
print("Sample các IEX Id missing:", missing_ids[:20])

Số lượng IEX Id trong iex_df nhưng không tìm thấy trong headcount master: 6
Sample các IEX Id missing: ['3046597' '3009086' '3090818' '3090108' '3090939' '3090140']


In [72]:
# Xuất ra file Excel, nếu đã có sẵn thì sẽ ghi đè
iex_df.to_excel("iex-data-extracted.xlsx", index=False)